# SIR outbreak sweep

In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

In [2]:
N = 1_000_000
I0 = 12
R0_grid = np.linspace(1.2, 3.4, 12)
gamma = 1/6.5

In [3]:
def sir(t, y, beta, gamma):
    S, I, R = y
    n = S + I + R
    return [-beta*S*I/n, beta*S*I/n - gamma*I, gamma*I]

In [4]:
def run(beta, days=240):
    y0 = [N - I0, I0, 0.0]
    sol = solve_ivp(sir, (0, days), y0, args=(beta, gamma), dense_output=True, max_step=1.0)
    return sol

In [5]:
def peak_of(sol):
    t = np.linspace(0, sol.t[-1], 2000)
    I = sol.sol(t)[1]
    return t[I.argmax()], I.max()

sweep below takes ~40s

In [6]:
results = {}
for r0 in R0_grid:
    beta = r0 * gamma
    results[r0] = run(beta)

In [7]:
peaks = {r0: peak_of(s) for r0, s in results.items()}

In [8]:
for r0, (tp, ip) in peaks.items():
    print(f'{r0:.2f}  peak day {tp:6.1f}  peak I {ip:,.0f}')

In [9]:
plt.figure(figsize=(11,5))
for r0, sol in results.items():
    t = np.linspace(0, 240, 800)
    plt.plot(t, sol.sol(t)[1], label=f'R0={r0:.1f}')
plt.legend(ncol=3)
plt.show()

In [10]:
def attack_rate(sol):
    return sol.sol(sol.t[-1])[2] / N

In [11]:
rates = {r0: attack_rate(s) for r0, s in results.items()}
rates

In [12]:
plt.plot(list(rates.keys()), list(rates.values()), 'o-')
plt.xlabel('R0'); plt.ylabel('attack rate')
plt.show()

In [13]:
def with_intervention(beta, start, factor=0.45, days=240):
    def rhs(t, y):
        b = beta * (factor if t >= start else 1.0)
        return sir(t, y, b, gamma)
    return solve_ivp(rhs, (0, days), [N-I0, I0, 0.0], dense_output=True, max_step=1.0)

In [14]:
iv = {d: with_intervention(2.4*gamma, d) for d in (20, 40, 60, 80)}

In [15]:
plt.figure(figsize=(11,5))
for d, sol in iv.items():
    t = np.linspace(0,240,800)
    plt.plot(t, sol.sol(t)[1], label=f'start day {d}')
plt.legend()
plt.show()

In [16]:
def summarise(sol):
    tp, ip = peak_of(sol)
    return {'peak_day': tp, 'peak_I': ip, 'attack': attack_rate(sol)}

In [17]:
import pandas as pd
pd.DataFrame({d: summarise(s) for d, s in iv.items()}).T

In [ ]:
np.save('sweep_peaks.npy', np.array([[r0, *peaks[r0]] for r0 in R0_grid]))

In [ ]:
def sensitivity(param, values):
    raise NotImplementedError